In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

from preprocess import build_fault_type_dataset, save_feature_list, COMMON_FEATURES

In [2]:
ARTIFACTS_DIR = Path("training/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "fault_type_model.pkl"
LABEL_ENCODER_PATH = ARTIFACTS_DIR / "fault_type_label_encoder.pkl"
FEATURES_PATH = ARTIFACTS_DIR / "fault_type_feature_list.json"
METRICS_PATH = ARTIFACTS_DIR / "fault_type_metrics.json"
TEST_CSV_PATH = ARTIFACTS_DIR / "fault_type_test_set.csv"
TEST_PREDICTIONS_CSV_PATH = ARTIFACTS_DIR / "fault_type_test_predictions.csv"


In [3]:
X, y = build_fault_type_dataset(only_failures=True)

print("Fault dataset shape:", X.shape)
print("\nFault type distribution BEFORE split:")
print(y.value_counts())

X.head()

Fault dataset shape: (339, 6)

Fault type distribution BEFORE split:
fault_type
HDF           115
PWF            91
OSF            78
TWF            46
NO_FAILURE      9
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
50,298.9,10.2,2861,4.6,1.378175,143
69,298.9,10.1,1410,65.7,9.700924,191
77,298.8,10.1,1455,41.3,6.292767,208
160,298.4,9.8,1282,60.7,8.149019,216
161,298.3,9.8,1412,52.3,7.733303,218


In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", list(label_encoder.classes_))

Classes: ['HDF', 'NO_FAILURE', 'OSF', 'PWF', 'TWF']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution BEFORE SMOTE:")
print(pd.Series(label_encoder.inverse_transform(y_train)).value_counts())

Train shape: (271, 6)
Test shape: (68, 6)

Train distribution BEFORE SMOTE:
HDF           92
PWF           73
OSF           62
TWF           37
NO_FAILURE     7
Name: count, dtype: int64


In [6]:
test_df = X_test.copy()
test_df["true_fault_type"] = label_encoder.inverse_transform(y_test)
test_df.to_csv(TEST_CSV_PATH, index=False)

print("Saved test set to:", TEST_CSV_PATH)
test_df.head()

Saved test set to: training\artifacts\fault_type_test_set.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],true_fault_type
5490,302.6,9.5,1288,68.5,9.239215,0,PWF
160,298.4,9.8,1282,60.7,8.149019,216,OSF
5394,302.8,9.5,1262,70.5,9.317021,234,PWF
4044,301.9,9.0,1419,47.7,7.088093,20,NO_FAILURE
4669,303.3,8.2,1373,47.3,6.800805,60,HDF


In [7]:
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Train shape AFTER SMOTE:", X_train_resampled.shape)
print("\nTrain distribution AFTER SMOTE:")
print(pd.Series(label_encoder.inverse_transform(y_train_resampled)).value_counts())

Train shape AFTER SMOTE: (460, 6)

Train distribution AFTER SMOTE:
HDF           92
OSF           92
PWF           92
TWF           92
NO_FAILURE    92
Name: count, dtype: int64


In [9]:
SAFE_FEATURE_MAP = {
    "Air temperature [K]": "air_temperature_k",
    "temp_diff": "temp_diff",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "power_kw": "power_kw",
    "Tool wear [min]": "tool_wear_min",
}

X_train_resampled = X_train_resampled.rename(columns=SAFE_FEATURE_MAP)
X_test = X_test.rename(columns=SAFE_FEATURE_MAP)

In [10]:
model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.5,
    min_child_weight=2,
    reg_alpha=0.05,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled, y_train_resampled)
print("Fault type model trained successfully")

Fault type model trained successfully


In [11]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

y_test_labels = label_encoder.inverse_transform(y_test)
y_pred_labels = label_encoder.inverse_transform(y_pred)

print("Classification report:")
print(classification_report(y_test_labels, y_pred_labels, digits=4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_labels, y_pred_labels, labels=label_encoder.classes_))

Classification report:
              precision    recall  f1-score   support

         HDF     0.9583    1.0000    0.9787        23
  NO_FAILURE     1.0000    1.0000    1.0000         2
         OSF     0.9333    0.8750    0.9032        16
         PWF     0.9474    1.0000    0.9730        18
         TWF     0.8750    0.7778    0.8235         9

    accuracy                         0.9412        68
   macro avg     0.9428    0.9306    0.9357        68
weighted avg     0.9397    0.9412    0.9395        68


Confusion matrix:
[[23  0  0  0  0]
 [ 0  2  0  0  0]
 [ 1  0 14  0  1]
 [ 0  0  0 18  0]
 [ 0  0  1  1  7]]


In [12]:
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

metrics = {
    "model_type": "fault_type_multiclass",
    "model_name": "XGBoost + SMOTE",
    "classes": list(label_encoder.classes_),
    "macro_f1": round(float(macro_f1), 6),
    "weighted_f1": round(float(weighted_f1), 6),
    "confusion_matrix": confusion_matrix(
        y_test_labels,
        y_pred_labels,
        labels=list(label_encoder.classes_)
    ).tolist(),
    "features": list(COMMON_FEATURES),
    "train_size_before_smote": int(len(X_train)),
    "train_size_after_smote": int(len(X_train_resampled)),
    "test_size": int(len(X_test)),
}

print(json.dumps(metrics, indent=2))

{
  "model_type": "fault_type_multiclass",
  "model_name": "XGBoost + SMOTE",
  "classes": [
    "HDF",
    "NO_FAILURE",
    "OSF",
    "PWF",
    "TWF"
  ],
  "macro_f1": 0.93569,
  "weighted_f1": 0.939522,
  "confusion_matrix": [
    [
      23,
      0,
      0,
      0,
      0
    ],
    [
      0,
      2,
      0,
      0,
      0
    ],
    [
      1,
      0,
      14,
      0,
      1
    ],
    [
      0,
      0,
      0,
      18,
      0
    ],
    [
      0,
      0,
      1,
      1,
      7
    ]
  ],
  "features": [
    "Air temperature [K]",
    "temp_diff",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "power_kw",
    "Tool wear [min]"
  ],
  "train_size_before_smote": 271,
  "train_size_after_smote": 460,
  "test_size": 68
}


In [13]:
proba_df = pd.DataFrame(
    y_prob,
    columns=[f"prob_{cls}" for cls in label_encoder.classes_]
)

test_predictions_df = X_test.reset_index(drop=True).copy()
test_predictions_df["true_fault_type"] = y_test_labels
test_predictions_df["predicted_fault_type"] = y_pred_labels
test_predictions_df = pd.concat([test_predictions_df, proba_df], axis=1)
test_predictions_df.to_csv(TEST_PREDICTIONS_CSV_PATH, index=False)

print("Saved test predictions to:", TEST_PREDICTIONS_CSV_PATH)
test_predictions_df.head()

Saved test predictions to: training\artifacts\fault_type_test_predictions.csv


,air_temperature_k,temp_diff,rotational_speed_rpm,torque_nm,power_kw,tool_wear_min,true_fault_type,predicted_fault_type,prob_HDF,prob_NO_FAILURE,prob_OSF,prob_PWF,prob_TWF
0,302.6,9.5,1288,68.5,9.239215,0,PWF,PWF,0.004149,0.005192,0.003933,0.984876,0.001849
1,298.4,9.8,1282,60.7,8.149019,216,OSF,OSF,0.002768,0.003396,0.982671,0.005072,0.006093
2,302.8,9.5,1262,70.5,9.317021,234,PWF,PWF,0.004645,0.003809,0.052032,0.933144,0.006370
3,301.9,9.0,1419,47.7,7.088093,20,NO_FAILURE,NO_FAILURE,0.009281,0.918819,0.019696,0.025060,0.027143
4,303.3,8.2,1373,47.3,6.800805,60,HDF,HDF,0.987058,0.008305,0.001553,0.001673,0.001411


In [14]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

,feature,importance
3,Torque [Nm],0.250801
1,temp_diff,0.214919
5,Tool wear [min],0.182444
4,power_kw,0.142026
2,Rotational speed [rpm],0.120706
0,Air temperature [K],0.089104


In [15]:
joblib.dump(model, MODEL_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_PATH)
save_feature_list(list(COMMON_FEATURES), FEATURES_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved label encoder to:", LABEL_ENCODER_PATH)
print("Saved feature list to:", FEATURES_PATH)
print("Saved metrics to:", METRICS_PATH)

Feature list saved to: training\artifacts\fault_type_feature_list.json
Saved model to: training\artifacts\fault_type_model.pkl
Saved label encoder to: training\artifacts\fault_type_label_encoder.pkl
Saved feature list to: training\artifacts\fault_type_feature_list.json
Saved metrics to: training\artifacts\fault_type_metrics.json
